# M3 Validation: Seed Trajectory Navigation
End-to-end M3 validation with deterministic seed walks and A3 latent injection.

In [ ]:
import time
import torch
from pathlib import Path
from IPython.display import display
from helpers.methods.access import infer_latent_shape
from helpers.methods.navigation import seed_walks
from helpers.runtime.pipeline import default_torch_dtype_for_device, load_flux_pipeline, resolve_runtime_device
from helpers.runtime.sampling import generate_with_prompt
from helpers.runtime.utility import flush_namespace

In [ ]:
output_dir = Path('../python/examples/navigation/outputs/m3_seed_trajectory_notebook').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

device = resolve_runtime_device()
dtype = default_torch_dtype_for_device(torch.float16, device)
try:
    pipe = load_flux_pipeline(
        torch_dtype=dtype,
        device=device,
        offload_mode="group",
        group_use_stream=True,
        group_low_cpu_mem_usage=False,
        enable_text_encoder_group_offload=True,
        text_encoder_group_offload_type="block_level",
        text_encoder_num_blocks_per_group=1,
        enable_vae_group_offload=True,
        vae_group_offload_type="leaf_level",
        vae_num_blocks_per_group=2,
        enable_vae_slicing=True,
        enable_vae_tiling=True,
        enable_layerwise_casting=True,
        layerwise_storage_dtype="float8_e4m3fn",
        layerwise_compute_dtype=dtype,
        enable_channels_last=True,
        enable_xformers_attention=True,
        enable_torch_compile=False,
    )

    shape = infer_latent_shape(height=1024, width=1024)
    trajectory = seed_walks(
        seed_a=5,
        seed_b=105,
        shape=shape,
        steps=5,
        method='slerp',
        dtype=dtype,
        device='cpu',
    )

    for i, latent in enumerate(trajectory):
        alpha = i / max(len(trajectory) - 1, 1)
        
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = generate_with_prompt(
                pipe=pipe,
                prompt='a forest trail after rain, natural lighting',
                seed=77,
                latents=latent,
                num_inference_steps=4,
                guidance_scale=0.0,
            )
        t1 = time.perf_counter()
        image = out.images[0]

        path = output_dir / f'm3_seedW_alpha_{alpha:.2f}_{(t1 - t0):.2f}.png'
        image.save(path)
        
        print(f'alpha={alpha:.2f} in {(t1 - t0):.2f} s -> {path}')
        display(image)
finally:
    flush_namespace(
        globals(),
        "pipe",
        "out",
        "image",
        "latent",
        verbose=True,
    )